# TRELLIS.2 - Bild zu 3D für OpenTTS

**Microsoft's Image-to-3D mit vollen PBR-Texturen (MIT Lizenz)**

**Workflow:**
1. Setup ausführen (einmal, ~5 min)
2. Bild hochladen
3. 3D-Modell generieren
4. Herunterladen

**Wichtig:** Benötigt A100 GPU (Colab Pro) - mindestens 24GB VRAM!

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DutchMaxwell/openTTS/blob/main/tools/3d-generation/trellis2_colab.ipynb)

---
## Schritt 1: Setup (einmal ausführen)
Dauert ca. 5 Minuten. Danach kannst du beliebig viele Bilder konvertieren.

In [ ]:
#@title 1. Setup ausführen {display-mode: "form"}

import torch
import os
import time
import sys
import threading
from IPython.display import display, HTML, clear_output

class LiveTimer:
    """Zeigt einen Live-Timer während langer Operationen"""
    def __init__(self, desc, estimate_min=5):
        self.desc = desc
        self.estimate_min = estimate_min
        self.running = False
        self.start_time = None
        
    def _timer_thread(self):
        while self.running:
            elapsed = time.time() - self.start_time
            mins = int(elapsed // 60)
            secs = int(elapsed % 60)
            estimate_str = f"~{self.estimate_min} min" if self.estimate_min else ""
            print(f"\r⏳ {self.desc} [{mins:02d}:{secs:02d}] {estimate_str}   ", end="", flush=True)
            time.sleep(1)
    
    def start(self):
        self.running = True
        self.start_time = time.time()
        self.thread = threading.Thread(target=self._timer_thread)
        self.thread.start()
        
    def stop(self):
        self.running = False
        self.thread.join()
        elapsed = time.time() - self.start_time
        mins = int(elapsed // 60)
        secs = int(elapsed % 60)
        print(f"\r✅ {self.desc} [{mins:02d}:{secs:02d}]              ")

def progress_bar(step, total, desc=""):
    """Zeigt einen Fortschrittsbalken an"""
    percent = int((step / total) * 100)
    bar_length = 25
    filled = int(bar_length * step / total)
    bar = "█" * filled + "░" * (bar_length - filled)
    print(f"[{bar}] {percent}% - {desc}")

# GPU Check
print("=" * 50)
print("GPU CHECK")
print("=" * 50)

if not torch.cuda.is_available():
    print("❌ KEINE GPU! Gehe zu: Runtime > Change runtime type > GPU")
    raise SystemExit("GPU required")

gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram:.1f} GB")

if vram < 23:
    print("\n❌ NICHT GENUG VRAM!")
    print("TRELLIS.2 benötigt mindestens 24GB (A100).")
    raise SystemExit("Insufficient VRAM")
else:
    print(f"✅ GPU OK")

# Installation
if not os.path.exists('/content/TRELLIS.2/trellis2'):
    total_start = time.time()
    
    print("\n" + "=" * 50)
    print("INSTALLATION (ca. 12-15 Minuten)")
    print("=" * 50 + "\n")
    
    # Step 1: Clone
    timer = LiveTimer("Repository klonen", estimate_min=None)
    timer.start()
    os.system("git clone --depth 1 https://github.com/microsoft/TRELLIS.2.git /content/TRELLIS.2 > /dev/null 2>&1")
    os.chdir("/content/TRELLIS.2")
    timer.stop()
    progress_bar(1, 6, "Repository ✓")
    
    # Step 2: Core packages
    timer = LiveTimer("Python-Pakete installieren", estimate_min=2)
    timer.start()
    os.system("pip install -q imageio imageio-ffmpeg tqdm easydict opencv-python-headless > /dev/null 2>&1")
    os.system("pip install -q ninja trimesh transformers gradio tensorboard pandas lpips > /dev/null 2>&1")
    os.system("pip install -q zstandard kornia timm pillow huggingface_hub einops safetensors accelerate diffusers > /dev/null 2>&1")
    timer.stop()
    progress_bar(2, 6, "Python-Pakete ✓")
    
    # Step 3: Flash attention (LONG!)
    timer = LiveTimer("Flash Attention kompilieren (CUDA)", estimate_min=10)
    timer.start()
    os.system("pip install flash-attn==2.7.3 --no-build-isolation > /dev/null 2>&1 || true")
    timer.stop()
    progress_bar(3, 6, "Flash Attention ✓")
    
    # Step 4: nvdiffrast
    timer = LiveTimer("CUDA Renderer kompilieren", estimate_min=3)
    timer.start()
    os.system("pip install git+https://github.com/NVlabs/nvdiffrast > /dev/null 2>&1")
    timer.stop()
    progress_bar(4, 6, "CUDA Renderer ✓")
    
    # Step 5: Submodules
    timer = LiveTimer("Extensions laden", estimate_min=None)
    timer.start()
    os.system("git submodule update --init --recursive > /dev/null 2>&1 || true")
    timer.stop()
    progress_bar(5, 6, "Extensions ✓")
    
    # Step 6: Finalize
    sys.path.insert(0, '/content/TRELLIS.2')
    progress_bar(6, 6, "Fertig!")
    
    total_elapsed = time.time() - total_start
    print(f"\n✅ Installation abgeschlossen in {total_elapsed/60:.1f} Minuten!")
else:
    os.chdir("/content/TRELLIS.2")
    sys.path.insert(0, '/content/TRELLIS.2')
    print("\n✅ TRELLIS.2 bereits installiert")

# Create folders
os.makedirs('/content/input', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)

print("\n" + "=" * 50)
print("✅ SETUP FERTIG - Weiter zu Schritt 2!")
print("=" * 50)

---
## Schritt 2: Bild hochladen

In [ ]:
#@title 2. Bild hochladen {display-mode: "form"}

from google.colab import files
from pathlib import Path
import shutil
from IPython.display import display, Image as IPImage

print("Wähle ein Bild aus (PNG oder JPG)...")
print("")

uploaded = files.upload()

if uploaded:
    for filename in uploaded.keys():
        dest = f'/content/input/{filename}'
        shutil.move(filename, dest)
        
        print(f"\n✅ Hochgeladen: {filename}")
        print("\nVorschau:")
        display(IPImage(dest, width=300))
        
        current_image = dest
        %store current_image
        
    print("\n" + "=" * 50)
    print("✅ BILD BEREIT - Weiter zu Schritt 3!")
    print("=" * 50)
else:
    print("❌ Kein Bild hochgeladen")

---
## Schritt 3: 3D-Modell generieren

In [ ]:
#@title 3. 3D-Modell generieren {display-mode: "form"}

#@markdown ### Einstellungen:
aufloesung = "512" #@param ["512", "1024", "1536"]

from pathlib import Path
from PIL import Image
import time
import sys
sys.path.insert(0, '/content/TRELLIS.2')

# Get uploaded image
%store -r current_image

try:
    img_path = Path(current_image)
except:
    images = list(Path('/content/input').glob('*.png')) + list(Path('/content/input').glob('*.jpg'))
    if images:
        img_path = images[-1]
    else:
        print("❌ Kein Bild gefunden! Führe zuerst Schritt 2 aus.")
        raise SystemExit()

output_path = f'/content/output/{img_path.stem}.glb'

print("=" * 50)
print(f"Eingabe: {img_path.name}")
print(f"Auflösung: {aufloesung}³")
print("=" * 50)

# Load pipeline
print("\n🔄 Lade TRELLIS.2 Modell...")
from trellis2.pipelines import Trellis2ImageTo3DPipeline

pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.to("cuda")

# Load image
image = Image.open(img_path).convert("RGB")

# Generate
print(f"\n🔄 Generiere 3D-Modell...")
start = time.time()

mesh = pipeline.run(
    image,
    resolution=int(aufloesung)
)[0]

# Export
mesh.export(output_path)

elapsed = time.time() - start

if Path(output_path).exists():
    size_mb = Path(output_path).stat().st_size / 1e6
    print("\n" + "=" * 50)
    print(f"✅ FERTIG in {elapsed:.0f} Sekunden!")
    print(f"📦 Datei: {Path(output_path).name} ({size_mb:.1f} MB)")
    print("=" * 50)
    print("\nWeiter zu Schritt 4 zum Herunterladen!")
    
    current_output = output_path
    %store current_output
else:
    print("\n❌ Fehler bei der Generierung")

---
## Schritt 4: Herunterladen

In [ ]:
#@title 4. GLB-Datei herunterladen {display-mode: "form"}

from google.colab import files
from pathlib import Path

%store -r current_output

try:
    output_path = Path(current_output)
except:
    outputs = list(Path('/content/output').glob('*.glb'))
    if outputs:
        output_path = outputs[-1]
    else:
        print("❌ Kein 3D-Modell gefunden! Führe zuerst Schritt 3 aus.")
        raise SystemExit()

if output_path.exists():
    size_mb = output_path.stat().st_size / 1e6
    print(f"📦 Download: {output_path.name} ({size_mb:.1f} MB)")
    print("")
    files.download(str(output_path))
    print("\n" + "=" * 50)
    print("✅ Download gestartet!")
    print("")
    print("Nächste Schritte:")
    print("1. GLB-Datei in OpenTTS importieren")
    print("2. Spawn > Import GLB")
    print("3. Positionieren und skalieren")
    print("4. L drücken zum Fixieren")
    print("=" * 50)
else:
    print("❌ Datei nicht gefunden")

---
## Weitere Bilder?

Gehe zurück zu **Schritt 2** und lade das nächste Bild hoch.

Das Modell bleibt geladen - weitere Generierungen sind schneller!

---
## Kein A100? Nutze den HuggingFace Space!

Kostenlos im Browser: **[TRELLIS.2 Demo](https://huggingface.co/spaces/microsoft/TRELLIS.2)**